# Notebook 2: Data Annotation

To validate model results, a ground truth dataset is needed. Since no labeled data exists, ground truth is created by manually cropping each image to isolate the lipstick color area, then extracting the average CIELAB color from the cropped region.

This notebook covers:
1. Selecting images for annotation via stratified sampling based on color taxonomy
2. Extracting the ground truth CIELAB color from the annotated (cropped) images

# Libraries

In [15]:
import glob
import os

from PIL import Image
from skimage.color import rgb2lab, lab2rgb
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Color Taxonomy for Stratified Sampling

The raw `parent_color` field contains 200+ unique values, many of which are near-duplicates differing only in word order (e.g. `pink nude` vs `nude pink`) or minor variations where one value is a subset of another (e.g. `burgundy` vs `burgundy wine` vs `wine burgundy`). Keeping these as-is would fragment the strata and undermine the stratified sampling — rare 1-count categories would be impossible to sample from meaningfully.

To address this, the `parent_color` values were consolidated into 18 simplified categories using a Claude-assisted color taxonomy. The grouping preserves meaningful color distinctions (e.g. separating `pink` from `pink_nude`) while collapsing redundant variations.

| Simplified Category | Original Values Included |
|---|---|
| **pink** | pink, hot pink, bubblegum, blush, petal, salmon |
| **red** | red, cherry red, ruby red, scarlet, crimson, tomato |
| **rose** | rose, rose pink, rosewood, rose nude, rose brown, rose mauve |
| **nude_beige** | nude, beige, nude beige, beige nude, clear, latte, champagne, tan |
| **pink_nude** | pink nude, nude pink, pink beige, brown nude pink, gray pink nude |
| **brown** | brown, chocolate, caramel, mocha, espresso, taupe, sienna, tawny |
| **berry_plum** | berry, plum, raspberry, burgundy, wine, cherry, blackberry, raisin, grape |
| **brick** | brick, brick red, brick brown, brick rose |
| **coral** | coral, coral pink, coral red, coral raspberry |
| **orange** | orange, orange red, terracotta, pumpkin, rust, apricot, copper, ochre |
| **mauve** | mauve, mauve pink, mauve nude, mauve brown, rose mauve, pink mauve |
| **peach** | peach, peach nude, peach pink, peach beige, peach bronze |
| **purple_fuchsia** | purple, violet, magenta, fuchsia, lilac, lavender, orchid |
| **gold_bronze** | gold, golden, bronze, golden bronze, yellow |
| **black** | black, ebony, charcoal |
| **blue_green** | blue, cobalt, navy, teal, turquoise, green, indigo, denim |
| **silver_gray** | gray, grey, silver, white, gunmetal |
| **other** | rare values with no clear color group |

# Data

In [16]:
import re
import unicodedata

BASE = os.path.abspath('../data')
DATA = os.path.join(BASE, 'processed')
RAW  = os.path.join(BASE, 'product_metadata')
IMGS = os.path.join(BASE, 'img', 'original')
IMGS_GT = os.path.join(BASE, 'img', 'groundtruth')

In [26]:
def slugify(text):
    if pd.isna(text): return ''
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', '_', text.strip())
    return text

# Load image metadata from notebook 1
df = pd.read_csv(f'{DATA}/products_with_images.csv')

# Keep only rows with a successfully downloaded image (valid filename has a '.')
df = df[df['img_name'].apply(lambda x: isinstance(x, str) and '.' in x)]
df = df.reset_index(drop=True)

# Merge parent_color from original metadata using id
meta = pd.read_csv(f'{RAW}/product_lipstick_metadata.csv', low_memory=False)
meta['id'] = (meta['category'].apply(slugify) + '__' +
              meta['brand'].apply(slugify) + '__' +
              meta['product'].apply(slugify) + '__' +
              meta['shade'].apply(slugify))

df = df.merge(meta[['id', 'parent_color']].drop_duplicates('id'), on='id', how='left')
print(f"Rows with images: {len(df)} | parent_color coverage: {df['parent_color'].notna().sum()} ({df['parent_color'].notna().mean()*100:.1f}%)")

Rows with images: 9502 | parent_color coverage: 6477 (68.2%)


In [27]:
def map_to_color_group(color):
    if pd.isna(color):
        return 'no_color_info'
    c = str(color).lower().strip()

    # Order matters: more specific patterns before broader ones
    if any(k in c for k in ['fuchsia', 'magenta', 'violet', 'purple', 'lilac', 'amethyst', 'orchid', 'lavender']):
        return 'purple_fuchsia'
    if any(k in c for k in ['pink nude', 'nude pink', 'pink beige', 'beige pink', 'pink brown nude', 'brown nude pink', 'gray pink nude', 'blush nude', 'blush pink']):
        return 'pink_nude'
    if any(k in c for k in ['berry', 'plum', 'burgundy', 'wine', 'cherry', 'blackberry', 'raspberry', 'raisin', 'cranberry', 'mulberry', 'sangria', 'merlot', 'cabernet', 'claret', 'currant', 'oxblood', 'maroon', 'boysenberry', 'pomegranate', 'prune', 'fig', 'aubergine', 'eggplant', 'garnet', 'marsala', 'bordeaux', 'grape']):
        return 'berry_plum'
    if any(k in c for k in ['brick']):
        return 'brick'
    if any(k in c for k in ['coral']):
        return 'coral'
    if any(k in c for k in ['orange', 'terracotta', 'pumpkin', 'rust', 'tangerine', 'persimmon', 'papaya', 'amber', 'apricot', 'ginger', 'saffron', 'cayenne', 'copper', 'vermilion', 'carrot', 'squash', 'clementine', 'melon', 'guava', 'grapefruit', 'ochre']):
        return 'orange'
    if any(k in c for k in ['peach']):
        return 'peach'
    if any(k in c for k in ['mauve']):
        return 'mauve'
    if any(k in c for k in ['rose', 'rosewood', 'rosewater', 'rosebud']):
        return 'rose'
    if any(k in c for k in ['brown', 'mocha', 'chocolate', 'cocoa', 'coffee', 'caramel', 'chestnut', 'hazelnut', 'toffee', 'espresso', 'sepia', 'mahogany', 'walnut', 'brunette', 'brandy', 'cognac', 'taupe', 'sienna', 'umber', 'tawny', 'truffle', 'sable', 'saddle', 'russet', 'malt', 'maple', 'cedar', 'chai', 'cinnamon', 'nutmeg', 'gingerbread', 'nougat', 'butterscotch', 'cashew', 'pecan', 'almond', 'fawn', 'buff', 'camel', 'mushroom', 'greige', 'griege', 'clay', 'auburn', 'pebble', 'corduroy', 'wood', 'rusk', 'redwood', 'desert']):
        return 'brown'
    if any(k in c for k in ['red', 'scarlet', 'crimson', 'ruby', 'tomato', 'poppy', 'geranium', 'hibiscus', 'cerise', 'apple']):
        return 'red'
    if any(k in c for k in ['pink', 'bubblegum', 'blush', 'petal', 'blossom', 'carnation', 'peony', 'salmon']):
        return 'pink'
    if any(k in c for k in ['nude', 'beige', 'clear', 'latte', 'champagne', 'sand', 'ivory', 'honey', 'ecru', 'biscuit', 'spice', 'tan']):
        return 'nude_beige'
    if any(k in c for k in ['gold', 'golden', 'bronze', 'yellow']):
        return 'gold_bronze'
    if any(k in c for k in ['black', 'ebony', 'charcoal', 'charcol']):
        return 'black'
    if any(k in c for k in ['blue', 'cobalt', 'navy', 'teal', 'turquoise', 'green', 'denim', 'indigo', 'spearmint', 'peacock', 'topaz']):
        return 'blue_green'
    if any(k in c for k in ['gray', 'grey', 'silver', 'white', 'gunmetal']):
        return 'silver_gray'

    return 'other'


df['color_group'] = df['parent_color'].apply(map_to_color_group)

In [28]:
counts = (df['color_group']
          .value_counts()
          .rename_axis('color_group')
          .reset_index(name='count'))
counts['%'] = (counts['count'] / counts['count'].sum() * 100).round(1)
counts

,color_group,count,%
0,no_color_info,3025,31.8
1,berry_plum,949,10.0
2,brown,736,7.7
3,red,705,7.4
4,rose,653,6.9
5,pink,597,6.3
6,mauve,460,4.8
7,nude_beige,436,4.6
8,purple_fuchsia,383,4.0
9,orange,376,4.0


In [29]:
print("Colors in the 'other' category:")
print(df[(df.color_group == 'other') & (df.parent_color.notna())].parent_color.to_string())

Colors in the 'other' category:
996            nectar
4762          heather
5715    dijon mustard
6094      wild azalea


# Stratified Sampling for Ground Truth Annotation

Sample size was calculated using `n = (Z² × σ²) / e²`, where:
- Z = 1.96 (95% confidence)
- σ = 13.97 (std of L channel from a prior lipstick study, n=191)
- e = 2 (margin of error in CIELAB units, approximately the just-noticeable difference threshold)

We used L as our σ in the sample size formula because it was the largest of the three CIELAB components (L=13.97, a=12.30, b=9.26), giving the most conservative (largest) sample size estimate.

This gives n ≈ 188, rounded up to **200**. Images with no color metadata (`no_color_info`) are excluded since the goal is to validate model accuracy across known color groups. We are prioritizing color coverage.

Allocation is **proportional** to group size with a **minimum of 5** per group. The final sample is **222** due to small groups being brought up to the minimum floor; the actual sampled total is slightly below the target allocation because some small groups (`other`, `silver_gray`) had fewer available images than their allocated minimum of 5.

In [30]:
N = 200
MIN = 5

# Exclude rows with no color metadata
df_known = df[df['color_group'] != 'no_color_info'].copy()

# Proportional allocation with minimum floor
group_counts = df_known['color_group'].value_counts()
allocation = (group_counts / group_counts.sum() * N).round().astype(int).clip(lower=MIN)

print(f"Total allocated: {allocation.sum()}")
print(allocation.reset_index().rename(columns={'color_group': 'color_group', 'count': 'allocation'}).to_string(index=False))

Total allocated: 223
   color_group  allocation
    berry_plum          29
         brown          23
           red          22
          rose          20
          pink          18
         mauve          14
    nude_beige          13
purple_fuchsia          12
        orange          12
     pink_nude          11
         peach          10
         coral           9
         brick           5
    blue_green           5
         black           5
   gold_bronze           5
   silver_gray           5
         other           5


In [31]:
# Sample from each group
sampled = (
    df_known.groupby('color_group', group_keys=False)
    .apply(lambda g: g.sample(n=min(allocation[g.name], len(g)), random_state=42))
)

print(f"Total sampled: {len(sampled)}")
sampled[['id', 'img_name', 'color_group', 'parent_color']].to_csv(f'{DATA}/annotation_sample.csv', index=False)
sampled[['id', 'img_name', 'color_group', 'parent_color']].head(10)

Total sampled: 222


/var/folders/x0/0hqgc7vn3ws2lx6klwmn4fj00000gn/T/ipykernel_43010/3506319723.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_known.groupby('color_group', group_keys=False)


,id,img_name,color_group,parent_color
1950,lipstick__ctzn_cosmetics__code_red_lipstick__pula,lipstick__ctzn_cosmetics__code_red_lipstick__p...,berry_plum,plum crimson red
6440,lipstick__nudestix__intense_matte_lip_cheek_pe...,lipstick__nudestix__intense_matte_lip_cheek_pe...,berry_plum,cranberry red
6689,lipstick__nyx_professional_makeup__shout_loud_...,lipstick__nyx_professional_makeup__shout_loud_...,berry_plum,wine red
7196,lipstick__queen_musia__skincare_supercharged_t...,lipstick__queen_musia__skincare_supercharged_t...,berry_plum,grape
7684,lipstick__romnd__dewyful_water_tint__07_cherry...,lipstick__romnd__dewyful_water_tint__07_cherry...,berry_plum,cherry red
7508,lipstick__revlon__super_lustrous_lipstick__812...,lipstick__revlon__super_lustrous_lipstick__812...,berry_plum,berry
8274,lipstick__soshe_beauty__ceramide_refillable_li...,lipstick__soshe_beauty__ceramide_refillable_li...,berry_plum,cherry
9234,lipstick__wonderskin__wonder_blading_peel_reve...,lipstick__wonderskin__wonder_blading_peel_reve...,berry_plum,burgundy red
4980,lipstick__mac_cosmetics__macximal_silky_matte_...,lipstick__mac_cosmetics__macximal_silky_matte_...,berry_plum,brown plum
1589,lipstick__clinique__pop_matte_lip_colour_prime...,lipstick__clinique__pop_matte_lip_colour_prime...,berry_plum,raspberry


# Copy Sampled Images to Ground Truth Folder

Copy the selected images from `img/original/` to `img/groundtruth/` so they can be manually cropped without modifying the originals.

In [32]:
import shutil

os.makedirs(IMGS_GT, exist_ok=True)

copied = 0
missing = 0

for img_name in sampled['img_name']:
    src = os.path.join(IMGS, img_name)
    dst = os.path.join(IMGS_GT, img_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing += 1

print(f"Copied: {copied} | Not found in original: {missing}")

Copied: 222 | Not found in original: 0


# Reading Data & Identifying Images with a Ground Truth Color Value

In [ ]:
import os
BASE    = os.path.abspath('../data')
DATA    = os.path.join(BASE, 'processed')
IMGS_GT = os.path.join(BASE, 'img', 'groundtruth')

I was not able to get the makeup color from all of the images because a number of images showed the container without the color of the actual makeup (e.g., lipstick). Thus, a small number of images will not have ground truth value.

To identify the data with ground truth value, I create a new variable in the metadata indicating coded 1 if there is a ground truth value and 0 if there is not. To do so, I use the list of cropped image files.

In [4]:
# Metadata
metadata = pd.read_csv(f'{DATA}/products_with_images.csv')
metadata.columns

Index(['Unnamed: 0', 'level_0', 'index', 'category', 'joined', 'brand',
       'product', 'shade', 'img_url', 'shade_description_original', 'id',
       'validation', 'img_name'],
      dtype='object')

In [36]:
# List of cropped images
folder_path = IMGS_GT
files = glob.glob(os.path.join(folder_path, '*'))  # List all files with full paths

# The file names without the extension because some extensions changed when the image was cropped
files = [os.path.splitext(os.path.basename(f))[0] for f in files]

# Print a few filenames
print(files[1:10])

['ulta250', 'ulta433', 's2427938-main-zoom', 'CF_PDP_Raunchy_swatch', 'juicypangwaterblusherCR01', 'Sunset', 'ulta8', 's2474427-av-04-zoom', 'ulta6']


In [10]:
# ground truth dummy
metadata['ground_truth'] = pd.Series(dtype='object')

# checking if each image has a cropped version
for i in range(len(metadata)):
  if metadata['img_name'][i].split('.')[0] in files:
    metadata.loc[i, 'ground_truth'] = 1
  else:
    metadata.loc[i, 'ground_truth'] = 0


Below, we observe that 88.8% of the images in the dataset have a corresponding ground truth value. The remaining images lack ground truth values, but this is due to factors unrelated to the color itself, such as incomplete data (e.g., the image had the packaging but did not show the makeup color.) Therefore, the absence of ground truth for these images should not significantly impact the overall analysis, as it doesn't introduce any bias related to the color properties being studied.

In [14]:
round(metadata.ground_truth.value_counts()/len(metadata)*100, 2)

,count
ground_truth,
1,88.8
0,11.2


# Ground Truth CIELAB Color Value

In [32]:
# store CIELAB color
metadata['ground_truth_CIELAB'] = pd.Series(dtype='object')

# extract color and save it for
for i in range(len(metadata)):
  if metadata['ground_truth'][i] == 1:
    # load image
    # file path
    directory = IMGS_GT
    filename = metadata['img_name'][i].split('.')[0]
    file_path = glob.glob(os.path.join(directory, filename + '.*'))

    # read image
    swatch = cv2.imread(file_path[0])

    # convert to Lab color space
    swatch = cv2.cvtColor(swatch, cv2.COLOR_BGR2RGB)
    img_lab = rgb2lab(swatch)

    # extract the average
    mean_swatch = img_lab.mean(axis=0).mean(axis=0)
    metadata.at[i, 'ground_truth_CIELAB'] = mean_swatch

In [34]:
metadata.ground_truth_CIELAB.info()

<class 'pandas.core.series.Series'>
RangeIndex: 527 entries, 0 to 526
Series name: ground_truth_CIELAB
Non-Null Count  Dtype 
--------------  ----- 
468 non-null    object
dtypes: object(1)
memory usage: 4.2+ KB


In [35]:
metadata.to_csv(f'{DATA}/ground_truth_labels.csv')